# 1. Ingest MEVA annotations and metadata

Fetch only the Kitware annotations and KRTD camera metadata from the MEVA repository. The temporary sparse clone is discarded; files are stored in `data/raw/`.

**Rerun behaviour** is controlled by `OVERWRITE`:

- `OVERWRITE = False` (default): if annotations and metadata are already staged, the MEVA clone is skipped entirely. If either folder is empty or missing, the repository is fetched and only files not already present are copied.
- `OVERWRITE = True`: both staged folders are deleted and everything is fetched again.

When run from `00_run_annotation_pipeline.ipynb`, the runner's `OVERWRITE` takes precedence.

In [3]:
import os
from pathlib import Path
import shutil
import subprocess
import tempfile
import time

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "notebooks").is_dir() and (parent / "annotation_pipeline").is_dir()
)
DATA_DIR = PROJECT_ROOT / "data" / "raw"
MEVA_REPOSITORY = "https://gitlab.kitware.com/meva/meva-data-repo.git"
KITWARE_PATH = Path("annotation/DIVA-phase-2/MEVA/kitware")
METADATA_PATH = Path("metadata/camera-models/krtd")
target_annotations = DATA_DIR / "annotations" / "kitware"
target_metadata = DATA_DIR / "metadata"

# True: delete the staged annotations/metadata and fetch everything again.
# False: skip the fetch when files are already staged; otherwise copy only missing files.
OVERWRITE = False
OVERWRITE = os.environ.get("PIPELINE_OVERWRITE", str(OVERWRITE)).strip().lower() in {"1", "true", "yes"}

print("=" * 72, flush=True)
print("Stage 1: ingesting Kitware annotations and camera metadata", flush=True)
print(f"OVERWRITE={OVERWRITE}; destination: {DATA_DIR}", flush=True)
print("=" * 72, flush=True)


def has_files(directory):
    return directory.is_dir() and any(path.is_file() for path in directory.rglob("*"))


def copy_if_missing(source_file, target_file):
    """Copy via a temporary file so an interrupted copy is never mistaken for a complete one."""
    if target_file.exists():
        return False
    target_file.parent.mkdir(parents=True, exist_ok=True)
    temporary_file = target_file.with_name(target_file.name + ".part")
    shutil.copy2(source_file, temporary_file)
    temporary_file.replace(target_file)
    return True


# The clip list is what notebook 2 needs; checking it (not just "any file") ignores unrelated
# folders such as the Validation scripts' meva_sparse cache that also live under kitware/.
annotations_staged = (target_annotations / "list-of-annotated-meva-clips.txt").is_file()
if not OVERWRITE and annotations_staged and has_files(target_metadata):
    print(f"OVERWRITE=False: annotations and metadata already staged in {DATA_DIR}; skipping MEVA fetch.")
else:
    with tempfile.TemporaryDirectory(prefix="meva-") as temporary_directory:
        repository_dir = Path(temporary_directory) / "meva-data-repo"
        for attempt in range(1, 4):
            try:
                subprocess.run(
                    [
                        "git", "-c", "http.version=HTTP/1.1", "clone", "--depth", "1",
                        "--filter=blob:none", "--sparse", MEVA_REPOSITORY, str(repository_dir),
                    ],
                    check=True,
                )
                subprocess.run(
                    [
                        "git", "-c", "http.version=HTTP/1.1", "-C", str(repository_dir),
                        "sparse-checkout", "set", str(KITWARE_PATH), str(METADATA_PATH),
                    ],
                    check=True,
                )
                break
            except subprocess.CalledProcessError:
                shutil.rmtree(repository_dir, ignore_errors=True)
                if attempt == 3:
                    raise
                time.sleep(attempt * 2)

        source_annotations = repository_dir / KITWARE_PATH
        source_metadata = repository_dir / METADATA_PATH
        if OVERWRITE:
            shutil.rmtree(target_annotations, ignore_errors=True)
            shutil.rmtree(target_metadata, ignore_errors=True)
        target_annotations.mkdir(parents=True, exist_ok=True)

        copied = skipped = 0
        seen_names = set()
        for source_file in sorted(path for path in source_annotations.rglob("*") if path.is_file()):
            if source_file.name in seen_names:
                raise FileExistsError(f"Cannot flatten Kitware files: {source_file.name}")
            seen_names.add(source_file.name)
            if copy_if_missing(source_file, target_annotations / source_file.name):
                copied += 1
            else:
                skipped += 1
        for source_file in sorted(path for path in source_metadata.rglob("*") if path.is_file()):
            if copy_if_missing(source_file, target_metadata / source_file.relative_to(source_metadata)):
                copied += 1
            else:
                skipped += 1

    print(f"OVERWRITE={OVERWRITE}: staged files in {DATA_DIR} ({copied} copied, {skipped} already present).")

Cloning into '/var/folders/0c/sn1ftfcs24l8fqqvjpwlpt1w0000gn/T/meva-icp16za6/meva-data-repo'...
Updating files: 100% (2/2), done.


Staged files in /Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/data/raw
